In [3]:
import numpy as np
from itertools import combinations

class PXClusterer:
    def __init__(self, max_cluster_size=4):
        self.max_cluster_size = max_cluster_size

    # ---------------------------
    # PX METRICS
    # ---------------------------

    def px2(self, pts):
        # 2-point group → perimeter is just Euclidean distance
        p1, p2 = pts
        return np.linalg.norm(p1 - p2)

    def px3(self, pts):
        # 3-point → triangle perimeter
        p1, p2, p3 = pts
        return (
            np.linalg.norm(p1 - p2)
            + np.linalg.norm(p2 - p3)
            + np.linalg.norm(p3 - p1)
        )

    def px4(self, pts):
        # 4-point → convex hull perimeter (simple)
        from scipy.spatial import ConvexHull

        hull = ConvexHull(pts)
        per = 0.0
        for i in range(len(hull.vertices)):
            a = pts[hull.vertices[i]]
            b = pts[hull.vertices[(i + 1) % len(hull.vertices)]]
            per += np.linalg.norm(a - b)
        return per

    # ---------------------------
    # MAIN CLUSTERING LOGIC
    # ---------------------------

    def cluster(self, points):
        """
        points = [(cls, u, v), ...]
        returns clusters = list of dictionaries:
        {
            'cluster_id': int,
            'cls': class_id,
            'centroid': (u_mean, v_mean),
            'points': [(cls,u,v), ...]
        }
        """

        points = np.array(points)  # shape (N,3)
        classes = np.unique(points[:, 0])

        final_clusters = []
        cluster_id = 0

        # Process each class independently
        for cls in classes:
            cls_points = points[points[:, 0] == cls][:, 1:].astype(float)
            N = len(cls_points)

            if N == 0:
                continue
            if N == 1:
                # One single cluster
                final_clusters.append({
                    "cluster_id": cluster_id,
                    "cls": cls,
                    "centroid": (cls_points[0][0], cls_points[0][1]),
                    "points": [(int(cls), cls_points[0][0], cls_points[0][1])]
                })
                cluster_id += 1
                continue

            # Step 1: For each point choose best PX group
            best_groups = []

            for i in range(N):
                best_px = float("inf")
                best_group = [i]  # default: single-point cluster

                # 2-point groups
                for j in range(N):
                    if j == i: 
                        continue
                    px = self.px2([cls_points[i], cls_points[j]])
                    if px < best_px:
                        best_px = px
                        best_group = [i, j]

                # 3-point groups
                if N >= 3:
                    for (j, k) in combinations(range(N), 2):
                        if i in (j, k):
                            continue
                        px = self.px3([cls_points[i], cls_points[j], cls_points[k]])
                        if px < best_px:
                            best_px = px
                            best_group = [i, j, k]

                # 4-point groups
                if N >= 4:
                    for (j, k, l) in combinations(range(N), 3):
                        if i in (j, k, l):
                            continue
                        px = self.px4([
                            cls_points[i], cls_points[j],
                            cls_points[k], cls_points[l]
                        ])
                        if px < best_px:
                            best_px = px
                            best_group = [i, j, k, l]

                best_groups.append(tuple(sorted(best_group)))

            # Step 2: Merge overlapping groups
            merged = []
            used = set()

            for g in best_groups:
                if any([idx in used for idx in g]):
                    continue
                merged.append(set(g))
                for idx in g:
                    used.add(idx)

            # Convert to final cluster output
            for g in merged:
                pts = cls_points[list(g)]
                centroid = pts.mean(axis=0)
                u_mean, v_mean = centroid

                final_clusters.append({
                    "cluster_id": cluster_id,
                    "cls": int(cls),
                    "centroid": (u_mean, v_mean),
                    "points": [
                        (int(cls), cls_points[i][0], cls_points[i][1])
                        for i in g
                    ]
                })
                cluster_id += 1

        return final_clusters


In [4]:
points = [
    (1, 10, 10),
    (1, 12, 11),
    (1, 11, 13),

    (1, 30, 30),
    (1, 31, 29),

    (1, 100, 100)
]

clusterer = PXClusterer()
result = clusterer.cluster(points)

for c in result:
    print("\nCluster:", c["cluster_id"])
    print("Class:", c["cls"])
    print("Centroid:", c["centroid"])
    print("Points:", c["points"])



Cluster: 0
Class: 1
Centroid: (np.float64(11.0), np.float64(10.5))
Points: [(1, np.float64(10.0), np.float64(10.0)), (1, np.float64(12.0), np.float64(11.0))]

Cluster: 1
Class: 1
Centroid: (np.float64(30.5), np.float64(29.5))
Points: [(1, np.float64(30.0), np.float64(30.0)), (1, np.float64(31.0), np.float64(29.0))]


In [13]:
import numpy as np
from itertools import combinations
from scipy.spatial import ConvexHull

class PXClusterer:
    def __init__(self, max_cluster_size=4):
        self.max_cluster_size = max_cluster_size

    # ---------------------------
    # PX METRICS
    # ---------------------------

    def px2(self, pts):
        p1, p2 = pts
        return np.linalg.norm(p1 - p2)

    def px3(self, pts):
        p1, p2, p3 = pts
        return (
            np.linalg.norm(p1 - p2)
            + np.linalg.norm(p2 - p3)
            + np.linalg.norm(p3 - p1)
        )

    def px4(self, pts):
        hull = ConvexHull(pts)
        per = 0.0
        for i in range(len(hull.vertices)):
            a = pts[hull.vertices[i]]
            b = pts[hull.vertices[(i + 1) % len(hull.vertices)]]
            per += np.linalg.norm(a - b)
        return per

    # ---------------------------
    # MAIN CLUSTERING LOGIC
    # ---------------------------

    def cluster(self, points):
        """
        points = [(cls, u, v), ...]
        returns list of cluster dictionaries
        """

        points = np.array(points)  # shape (N,3)
        classes = np.unique(points[:, 0])

        final_clusters = []
        cluster_id = 0

        # Process each class separately
        for cls in classes:
            cls_points = points[points[:, 0] == cls][:, 1:].astype(float)
            N = len(cls_points)

            if N == 0:
                continue

            if N == 1:
                final_clusters.append({
                    "cluster_id": cluster_id,
                    "cls": int(cls),
                    "centroid": (float(cls_points[0][0]), float(cls_points[0][1])),
                    "points": [(int(cls), float(cls_points[0][0]), float(cls_points[0][1]))]
                })
                cluster_id += 1
                continue

            # Step 1: For each point, choose best PX group
            best_groups = []

            for i in range(N):
                best_px = float("inf")
                best_group = [i]

                # 2-point groups
                for j in range(N):
                    if j == i:
                        continue
                    px = self.px2([cls_points[i], cls_points[j]])
                    if px < best_px:
                        best_px = px
                        best_group = [i, j]

                # 3-point groups
                if N >= 3:
                    for (j, k) in combinations(range(N), 2):
                        if i in (j, k):
                            continue
                        px = self.px3([cls_points[i], cls_points[j], cls_points[k]])
                        if px < best_px:
                            best_px = px
                            best_group = [i, j, k]

                # 4-point groups
                if N >= 4:
                    for (j, k, l) in combinations(range(N), 3):
                        if i in (j, k, l):
                            continue
                        px = self.px4([
                            cls_points[i], cls_points[j],
                            cls_points[k], cls_points[l]
                        ])
                        if px < best_px:
                            best_px = px
                            best_group = [i, j, k, l]

                best_groups.append(tuple(sorted(best_group)))

            # Step 2: Merge overlapping groups
            # Step 2: Proper merging of overlapping groups
            merged = []

            for g in best_groups:
                g = set(g)
                merged_into_existing = False

                for m in merged:
                    if not g.isdisjoint(m):   # overlap found
                        m.update(g)
                        merged_into_existing = True
                        break

                if not merged_into_existing:
                    merged.append(set(g))

            # Step 3: Build final clusters with centroids
            for g in merged:
                pts = cls_points[list(g)]
                centroid = pts.mean(axis=0)
                u_mean, v_mean = float(centroid[0]), float(centroid[1])

                final_clusters.append({
                    "cluster_id": cluster_id,
                    "cls": int(cls),
                    "centroid": (u_mean, v_mean),
                    "points": [
                        (int(cls), float(cls_points[i][0]), float(cls_points[i][1]))
                        for i in g
                    ]
                })
                cluster_id += 1

        return final_clusters


In [18]:
# points = [
#     (0,1622, 1272),
#     (0,1629, 1227),
#     (0,1645, 1160),
#     (0,1652, 1120)
# ]
points = [(0, 1653.7520030888882, 1285.875381892927), (0, 1724.301603169276, 1124.92314239387), (1, 1245.050107337042, 1543.085390495481), (0, 1738.8202702078834, 1098.500768306029), (0, 1687.151223100957, 1244.0465198221214)]
clusterer = PXClusterer()
clusters = clusterer.cluster(points)

for c in clusters:
    print("\nCluster:", c["cluster_id"])
    print("Class:", c["cls"])
    print("Centroid:", c["centroid"])
    print("Points:", c["points"])



Cluster: 0
Class: 0
Centroid: (1731.5609366885797, 1111.7119553499494)
Points: [(0, 1724.301603169276, 1124.92314239387), (0, 1738.8202702078834, 1098.500768306029)]

Cluster: 1
Class: 0
Centroid: (1670.4516130949225, 1264.960950857524)
Points: [(0, 1653.7520030888882, 1285.875381892927), (0, 1687.151223100957, 1244.0465198221214)]

Cluster: 2
Class: 1
Centroid: (1245.050107337042, 1543.085390495481)
Points: [(1, 1245.050107337042, 1543.085390495481)]


In [15]:
import numpy as np
from itertools import combinations
from scipy.spatial import ConvexHull

class PXClusterer:
    def __init__(self, max_cluster_size=4):
        self.max_cluster_size = max_cluster_size

    # ---------------------------
    # PX METRICS
    # ---------------------------

    def px2(self, pts):
        p1, p2 = pts
        return np.linalg.norm(p1 - p2)

    def px3(self, pts):
        p1, p2, p3 = pts
        return (
            np.linalg.norm(p1 - p2)
            + np.linalg.norm(p2 - p3)
            + np.linalg.norm(p3 - p1)
        )

    def px4(self, pts):
        hull = ConvexHull(pts)
        per = 0.0
        for i in range(len(hull.vertices)):
            a = pts[hull.vertices[i]]
            b = pts[hull.vertices[(i + 1) % len(hull.vertices)]]
            per += np.linalg.norm(a - b)
        return per

    # ---------------------------
    # MAIN CLUSTERING LOGIC
    # ---------------------------

    def cluster(self, points):
        """
        points = [(cls, u, v), ...]
        returns list of cluster dictionaries
        """

        points = np.array(points)  # shape (N,3)
        classes = np.unique(points[:, 0])

        final_clusters = []
        cluster_id = 0

        # Process each class separately
        for cls in classes:
            cls_points = points[points[:, 0] == cls][:, 1:].astype(float)
            N = len(cls_points)

            if N == 0:
                continue

            if N == 1:
                final_clusters.append({
                    "cluster_id": cluster_id,
                    "cls": int(cls),
                    "centroid": (float(cls_points[0][0]), float(cls_points[0][1])),
                    "points": [(int(cls), float(cls_points[0][0]), float(cls_points[0][1]))]
                })
                cluster_id += 1
                continue

            # Step 1: For each point choose its BEST PX group
            px_choice = []

            for i in range(N):
                best_px = float("inf")
                best_group = [i]

                # 2-point groups
                for j in range(N):
                    if j == i:
                        continue
                    px = self.px2([cls_points[i], cls_points[j]])
                    if px < best_px:
                        best_px = px
                        best_group = [i, j]

                # 3-point groups
                if N >= 3:
                    for (j, k) in combinations(range(N), 2):
                        if i in (j, k):
                            continue
                        px = self.px3([cls_points[i], cls_points[j], cls_points[k]])
                        if px < best_px:
                            best_px = px
                            best_group = [i, j, k]

                # 4-point groups
                if N >= 4:
                    for (j, k, l) in combinations(range(N), 3):
                        if i in (j, k, l):
                            continue
                        px = self.px4([
                            cls_points[i], cls_points[j],
                            cls_points[k], cls_points[l]
                        ])
                        if px < best_px:
                            best_px = px
                            best_group = [i, j, k, l]

                px_choice.append(tuple(sorted(best_group)))

            # ---------------------------------------
            # STEP 2: CONSENSUS MERGING (REAL FIX)
            # ---------------------------------------
            # Keep only groups where all members chose the SAME group.
            valid_groups = []

            for group in set(px_choice):
                members = list(group)

                # Check if every member selected THIS group exactly
                if all(px_choice[m] == group for m in members):
                    valid_groups.append(set(members))

            # Remove nested or duplicate clusters
            cleaned = []
            for g in valid_groups:
                if not any(g < other for other in valid_groups if g != other):
                    cleaned.append(g)

            # STEP 3: Create final clusters for this class
            for g in cleaned:
                pts = cls_points[list(g)]
                centroid = pts.mean(axis=0)
                u_mean, v_mean = float(centroid[0]), float(centroid[1])

                final_clusters.append({
                    "cluster_id": cluster_id,
                    "cls": int(cls),
                    "centroid": (u_mean, v_mean),
                    "points": [
                        (int(cls), float(cls_points[i][0]), float(cls_points[i][1]))
                        for i in g
                    ]
                })
                cluster_id += 1

            # STEP 4: Handle any leftover points as singles
            used = set().union(*cleaned)
            unused_points = [i for i in range(N) if i not in used]

            for i in unused_points:
                final_clusters.append({
                    "cluster_id": cluster_id,
                    "cls": int(cls),
                    "centroid": (float(cls_points[i][0]), float(cls_points[i][1])),
                    "points": [(int(cls), float(cls_points[i][0]), float(cls_points[i][1]))]
                })
                cluster_id += 1

        return final_clusters


In [16]:
points = [
    (0,1622, 1272),
    (0,1629, 1227),
    (0,1645, 1160),
    (0,1652, 1120)
]
clusterer = PXClusterer()
clusters = clusterer.cluster(points)

for c in clusters:
    print("\nCluster:", c["cluster_id"])
    print("Class:", c["cls"])
    print("Centroid:", c["centroid"])
    print("Points:", c["points"])



Cluster: 0
Class: 0
Centroid: (1625.5, 1249.5)
Points: [(0, 1622.0, 1272.0), (0, 1629.0, 1227.0)]

Cluster: 1
Class: 0
Centroid: (1648.5, 1140.0)
Points: [(0, 1645.0, 1160.0), (0, 1652.0, 1120.0)]


In [5]:
import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree
from itertools import combinations


class MSTMax4Clusterer:
    def __init__(self, max_cluster_size=4):
        self.max_size = max_cluster_size

    def _get_components(self, edges, N):
        """Return connected components from adjacency."""
        visited = set()
        comps = []

        for i in range(N):
            if i not in visited:
                stack = [i]
                group = []
                while stack:
                    v = stack.pop()
                    if v in visited:
                        continue
                    visited.add(v)
                    group.append(v)
                    for u in range(N):
                        if edges[v][u] or edges[u][v]:
                            stack.append(u)
                comps.append(group)
        return comps

    def cluster(self, points):
        """
        points = [(cls, u, v), ...]
        return clusters with centroid and class
        """
        points = np.array(points)
        classes = np.unique(points[:, 0])
        final_clusters = []
        cid = 0

        for cls in classes:
            cls_pts = points[points[:, 0] == cls][:, 1:].astype(float)
            N = len(cls_pts)

            if N == 0:
                continue

            if N == 1:
                p = cls_pts[0]
                final_clusters.append({
                    "cluster_id": cid,
                    "cls": int(cls),
                    "centroid": (float(p[0]), float(p[1])),
                    "points": [(int(cls), float(p[0]), float(p[1]))]
                })
                cid += 1
                continue

            # Pairwise distance matrix
            D = np.linalg.norm(cls_pts[:, None, :] - cls_pts[None, :, :], axis=2)

            # Build MST
            mst = minimum_spanning_tree(D).toarray()

            # Build full adjacency (MST is directed from scipy)
            adj = (mst + mst.T)

            # Sort edges by weight descending → largest edges first
            edges_list = []
            for i in range(N):
                for j in range(N):
                    if adj[i][j] > 0:
                        edges_list.append((i, j, adj[i][j]))
            edges_list.sort(key=lambda x: x[2], reverse=True)

            # Working adjacency copy
            work_adj = (adj > 0).astype(int)

            # Split until all components are <= max_size
            for (i, j, w) in edges_list:
                comps = self._get_components(work_adj, N)
                if all(len(c) <= self.max_size for c in comps):
                    break

                # remove largest edge
                work_adj[i][j] = 0
                work_adj[j][i] = 0

            # Final components
            comps = self._get_components(work_adj, N)

            # Convert each component into output cluster
            for comp in comps:
                pts = cls_pts[comp]
                centroid = pts.mean(axis=0)
                u, v = float(centroid[0]), float(centroid[1])

                final_clusters.append({
                    "cluster_id": cid,
                    "cls": int(cls),
                    "centroid": (u, v),
                    "points": [
                        (int(cls), float(cls_pts[i][0]), float(cls_pts[i][1]))
                        for i in comp
                    ]
                })
                cid += 1

        return final_clusters


In [9]:
points = [
    (0,1622, 1272),
    (0,1629, 1227),
    (0,1645, 1160),
    (0,1652, 1120)
]

clusterer = MSTMax4Clusterer()
clusters = clusterer.cluster(points)

for c in clusters:
    print("\nCluster:", c["cluster_id"])
    print("Class:", c["cls"])
    print("Centroid:", c["centroid"])
    print("Points:", c["points"])



Cluster: 0
Class: 0
Centroid: (1637.0, 1194.75)
Points: [(0, 1622.0, 1272.0), (0, 1629.0, 1227.0), (0, 1645.0, 1160.0), (0, 1652.0, 1120.0)]


In [10]:
import numpy as np
from itertools import combinations

class RatioGreedyClusterer:
    def __init__(self, alpha=2.5, max_size=4):
        self.alpha = alpha
        self.max_size = max_size

    def cluster(self, points):
        points = np.array(points)
        classes = np.unique(points[:,0])
        final_clusters = []
        cid = 0

        for cls in classes:
            cls_pts = points[points[:,0] == cls][:,1:].astype(float)
            N = len(cls_pts)

            used = set()
            all_idx = list(range(N))

            if N == 1:
                p = cls_pts[0]
                final_clusters.append({
                    "cluster_id": cid,
                    "cls": int(cls),
                    "centroid": (float(p[0]), float(p[1])),
                    "points": [(int(cls), float(p[0]), float(p[1]))],
                })
                cid += 1
                continue

            # Precompute distances
            D = np.linalg.norm(cls_pts[:,None,:] - cls_pts[None,:,:], axis=2)

            while len(used) < N:
                remaining = [i for i in all_idx if i not in used]

                # If only one left → single cluster
                if len(remaining) == 1:
                    i = remaining[0]
                    p = cls_pts[i]
                    final_clusters.append({
                        "cluster_id": cid,
                        "cls": int(cls),
                        "centroid": (float(p[0]), float(p[1])),
                        "points": [(int(cls), float(p[0]), float(p[1]))],
                    })
                    cid += 1
                    used.add(i)
                    continue

                # -----------------------------------------
                # Step 1: Find least-distance pair (i,j)
                # -----------------------------------------
                best_pair = None
                best_dist = float("inf")

                for i,j in combinations(remaining, 2):
                    if D[i,j] < best_dist:
                        best_dist = D[i,j]
                        best_pair = (i,j)

                # Start cluster
                cluster = list(best_pair)
                used.update(cluster)

                #------------------------------------------
                # Step 2: Try adding a third point
                #------------------------------------------
                if len(cluster) < self.max_size:
                    best_k = None
                    best_d = float("inf")
                    i,j = cluster

                    for k in remaining:
                        if k in cluster:
                            continue
                        # must be close to BOTH pair members
                        if D[i,k] <= self.alpha * best_dist and D[j,k] <= self.alpha * best_dist:
                            d_avg = (D[i,k] + D[j,k]) / 2
                            if d_avg < best_d:
                                best_d = d_avg
                                best_k = k

                    if best_k is not None:
                        cluster.append(best_k)
                        used.add(best_k)

                #------------------------------------------
                # Step 3: Try adding a 4th point
                #------------------------------------------
                if len(cluster) < self.max_size:
                    i,j = cluster[0], cluster[1]
                    pair_dist = D[i,j]

                    best_k = None
                    best_d = float("inf")

                    for k in remaining:
                        if k in cluster:
                            continue
                        if all(D[p,k] <= self.alpha * pair_dist for p in cluster):
                            d_avg = np.mean([D[p,k] for p in cluster])
                            if d_avg < best_d:
                                best_d = d_avg
                                best_k = k

                    if best_k is not None:
                        cluster.append(best_k)
                        used.add(best_k)

                #------------------------------------------
                # Save cluster
                #------------------------------------------
                pts = cls_pts[cluster]
                centroid = pts.mean(axis=0)

                final_clusters.append({
                    "cluster_id": cid,
                    "cls": int(cls),
                    "centroid": (float(centroid[0]), float(centroid[1])),
                    "points": [
                        (int(cls), float(cls_pts[i][0]), float(cls_pts[i][1]))
                        for i in cluster
                    ],
                })
                cid += 1

        return final_clusters


In [11]:
points = [
    (0,1622, 1272),
    (0,1629, 1227),
    (0,1645, 1160),
    (0,1652, 1120)
]

cl = RatioGreedyClusterer(alpha=2.5)
clusters = cl.cluster(points)

for c in clusters:
    print("\nCluster:", c["cluster_id"])
    print("Class:", c["cls"])
    print("Centroid:", c["centroid"])
    print("Points:", c["points"])



Cluster: 0
Class: 0
Centroid: (1648.5, 1140.0)
Points: [(0, 1645.0, 1160.0), (0, 1652.0, 1120.0)]

Cluster: 1
Class: 0
Centroid: (1625.5, 1249.5)
Points: [(0, 1622.0, 1272.0), (0, 1629.0, 1227.0)]


In [20]:
import numpy as np
from itertools import combinations

class PairFirstClusterer:
    def __init__(self, alpha=2.5, max_size=4):
        self.alpha = alpha
        self.max_size = max_size

    def cluster(self, points):
        points = np.array(points)
        classes = np.unique(points[:, 0])
        final_clusters = []
        cid = 0

        for cls in classes:
            cls_pts = points[points[:, 0] == cls][:, 1:].astype(float)
            N = len(cls_pts)
            used = set()
            ids = list(range(N))

            if N == 1:
                p = cls_pts[0]
                final_clusters.append({
                    "cluster_id": cid,
                    "cls": int(cls),
                    "centroid": (float(p[0]), float(p[1])),
                    "points": [(int(cls), float(p[0]), float(p[1]))]
                })
                cid += 1
                continue

            D = np.linalg.norm(cls_pts[:, None, :] - cls_pts[None, :, :], axis=2)

            while len(used) < N:
                remaining = [i for i in ids if i not in used]

                if len(remaining) == 1:
                    i = remaining[0]
                    p = cls_pts[i]
                    final_clusters.append({
                        "cluster_id": cid,
                        "cls": int(cls),
                        "centroid": (float(p[0]), float(p[1])),
                        "points": [(int(cls), float(p[0]), float(p[1]))]
                    })
                    cid += 1
                    used.add(i)
                    continue

                # STEP 1: Find closest pair
                best_pair = None
                best_dist = float("inf")

                for i, j in combinations(remaining, 2):
                    if D[i, j] < best_dist:
                        best_dist = D[i, j]
                        best_pair = (i, j)

                i, j = best_pair
                cluster = [i, j]
                used.update(cluster)

                pair_dist = best_dist   # <-- reference distance forever

                # STEP 2: Try adding third and fourth using STRICT rule
                while len(cluster) < self.max_size:
                    best_k = None
                    best_local = float("inf")

                    for k in remaining:
                        if k in cluster:
                            continue

                        # STRICT: must be close to BOTH pair members
                        if D[i, k] <= self.alpha * pair_dist and D[j, k] <= self.alpha * pair_dist:
                            d_sum = D[i, k] + D[j, k]
                            if d_sum < best_local:
                                best_local = d_sum
                                best_k = k

                    if best_k is not None:
                        cluster.append(best_k)
                        used.add(best_k)
                    else:
                        break

                # SAVE CLUSTER
                pts = cls_pts[cluster]
                centroid = pts.mean(axis=0)

                final_clusters.append({
                    "cluster_id": cid,
                    "cls": int(cls),
                    "centroid": (float(centroid[0]), float(centroid[1])),
                    "points": [
                        (int(cls), float(cls_pts[x][0]), float(cls_pts[x][1]))
                        for x in cluster
                    ]
                })

                cid += 1

        return final_clusters


In [22]:
points = [
    (1, 1675.136, 1308.594),
    (1, 1712.296, 1201.402),

    (0, 1616.3547, 1284.4453),
    (0, 1629.8689, 1226.636),
    (0, 1680.0, 1250.0),
    (0, 1690.0, 1253.0),
    (0, 1710.0, 1260.0),
    (0, 1000.0, 1000.0),
]

cl = PairFirstClusterer(alpha=2.0)
clusters = cl.cluster(points)

for c in clusters:
    print("\nCluster:", c["cluster_id"])
    print("Class:", c["cls"])
    print("Centroid:", c["centroid"])
    print("Points:", c["points"])



Cluster: 0
Class: 0
Centroid: (1685.0, 1251.5)
Points: [(0, 1680.0, 1250.0), (0, 1690.0, 1253.0)]

Cluster: 1
Class: 0
Centroid: (1652.0745333333334, 1257.0271)
Points: [(0, 1616.3547, 1284.4453), (0, 1629.8689, 1226.636), (0, 1710.0, 1260.0)]

Cluster: 2
Class: 0
Centroid: (1000.0, 1000.0)
Points: [(0, 1000.0, 1000.0)]

Cluster: 3
Class: 1
Centroid: (1693.716, 1254.998)
Points: [(1, 1675.136, 1308.594), (1, 1712.296, 1201.402)]
